# SH17 PP-PicoDet-L Portable Benchmark

This notebook benchmarks PP-PicoDet-L on SH17 as a non-YOLO, non-EfficientDet lightweight detector. PP-PicoDet-L has about 5.80M parameters, below the 7.2M parameter budget of YOLOv9s.

The default experiment sequence now uses a faster, measured augmentation profile for the Windows 4060 Ti training machine:

1. `picodet_l_320_baseline_fast_20e`
2. `picodet_l_640_scale_fast_20e`
3. `picodet_l_640_oversample_fast_20e`
4. `picodet_l_640_zoom_crop_fast_20e`

Run `RUN_SPEED_BENCHMARK=True` first if data loading is slow. The benchmark compares `official`, `no_distort`, `fast`, and `fixed` augmentation profiles on a 960-image subset, then writes `speed_benchmark.csv`.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()


def _env_flag(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return bool(default)
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_TRAINING = _env_flag("SH17_RUN_TRAINING", False)
RUN_SPEED_BENCHMARK = _env_flag("SH17_RUN_SPEED_BENCHMARK", True)
USE_GPU = _env_flag("SH17_USE_GPU", True)
ALLOW_CPU_FALLBACK = os.environ.get("SH17_ALLOW_CPU_FALLBACK")
if ALLOW_CPU_FALLBACK is None:
    # Local smoke checks can continue on CPU; full training should fail loudly if GPU is unavailable.
    ALLOW_CPU_FALLBACK = not RUN_TRAINING
else:
    ALLOW_CPU_FALLBACK = ALLOW_CPU_FALLBACK.strip().lower() in {"1", "true", "yes", "y", "on"}

TRAIN_WITH_AMP = _env_flag("SH17_TRAIN_WITH_AMP", True)
TRAIN_AUG_PROFILE = os.environ.get("SH17_TRAIN_AUG_PROFILE", "fast")
VERIFY_PADDLE_GPU_RUNTIME = _env_flag("SH17_VERIFY_PADDLE_GPU_RUNTIME", True)
WORKER_NUM = int(os.environ.get("SH17_WORKER_NUM", "6"))
USE_SHARED_MEMORY = _env_flag("SH17_USE_SHARED_MEMORY", os.name != "nt")

SEED = int(os.environ.get("SH17_SEED", "0"))
PILOT_EPOCHS = int(os.environ.get("SH17_PILOT_EPOCHS", "20"))
FULL_EPOCHS = int(os.environ.get("SH17_FULL_EPOCHS", "50"))
EPOCHS = int(os.environ.get("SH17_EPOCHS", str(PILOT_EPOCHS)))
SNAPSHOT_EPOCH = int(os.environ.get("SH17_SNAPSHOT_EPOCH", "5"))
BENCHMARK_IMAGE_LIMIT = int(os.environ.get("SH17_BENCHMARK_IMAGE_LIMIT", "960"))


def _data_root_candidates():
    candidates = [
        Path("C:/data/SH17"),
        Path("E:/data/SH17"),
        Path("D:/data/SH17"),
        PROJECT_ROOT,
        PROJECT_ROOT / "dataset",
        PROJECT_ROOT / "datasets",
        PROJECT_ROOT / "datasets" / "sh17-dataset-for-ppe-detection",
        PROJECT_ROOT / "datasets" / "mugheesahmad" / "sh17-dataset-for-ppe-detection",
        PROJECT_ROOT / "data",
        PROJECT_ROOT / "data" / "sh17-dataset-for-ppe-detection",
        PROJECT_ROOT / "data" / "mugheesahmad" / "sh17-dataset-for-ppe-detection",
        PROJECT_ROOT.parent / "sh17-dataset-for-ppe-detection",
        PROJECT_ROOT.parent / "SH17_dataset",
        PROJECT_ROOT.parent / "dataset",
        PROJECT_ROOT.parent / "datasets" / "sh17-dataset-for-ppe-detection",
        Path("/kaggle/input/sh17-dataset-for-ppe-detection"),
        Path("/kaggle/input/datasets/mugheesahmad/sh17-dataset-for-ppe-detection"),
    ]
    return list(dict.fromkeys(path.expanduser() for path in candidates))


def _has_manifest_pair(root):
    return (root / "train_files.txt").exists() and (root / "val_files.txt").exists()


def _resolve_data_root():
    env_data_root = os.environ.get("SH17_DATA_ROOT")
    if env_data_root:
        return Path(env_data_root).expanduser()
    for candidate in DATA_ROOT_CANDIDATES:
        if _has_manifest_pair(candidate):
            return candidate
    return PROJECT_ROOT


DATA_ROOT_CANDIDATES = _data_root_candidates()
DATA_ROOT = _resolve_data_root()
DATA_ROOT = Path(DATA_ROOT).expanduser()
TRAIN_MANIFEST = Path(os.environ.get("SH17_TRAIN_MANIFEST", DATA_ROOT / "train_files.txt")).expanduser()
VAL_MANIFEST = Path(os.environ.get("SH17_VAL_MANIFEST", DATA_ROOT / "val_files.txt")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("SH17_OUTPUT_ROOT", PROJECT_ROOT / "outputs" / "picodet_l_paddledet")).expanduser()
PADDLEDET_ROOT = Path(os.environ.get("PADDLEDET_ROOT", OUTPUT_ROOT / "PaddleDetection")).expanduser()

AUTO_SETUP = _env_flag("SH17_AUTO_SETUP", True)
INSTALL_PADDLEDET_REQUIREMENTS = _env_flag("SH17_INSTALL_PADDLEDET_REQUIREMENTS", True)
PADDLEDET_REQUIREMENTS_ONLY_BINARY = _env_flag("SH17_PADDLEDET_REQUIREMENTS_ONLY_BINARY", os.name == "nt")
PADDLEDET_BINARY_ONLY_PACKAGES = ["numpy", "scipy", "opencv-python", "pycocotools", "shapely"]
BOOTSTRAP_PIP_PACKAGES = ["setuptools<81", "wheel"]
DEFAULT_PADDLE_PACKAGE = "paddlepaddle-gpu" if USE_GPU and not ALLOW_CPU_FALLBACK else "paddlepaddle"
DEFAULT_PADDLE_INDEX_URL = "https://www.paddlepaddle.org.cn/packages/stable/cu118/" if USE_GPU and not ALLOW_CPU_FALLBACK else ""
PADDLE_INSTALL_PACKAGE = os.environ.get("PADDLE_INSTALL_PACKAGE", DEFAULT_PADDLE_PACKAGE)
PADDLE_INSTALL_INDEX_URL = os.environ.get("PADDLE_INSTALL_INDEX_URL", DEFAULT_PADDLE_INDEX_URL)
AUTO_REINSTALL_PADDLE_PACKAGE = _env_flag("SH17_AUTO_REINSTALL_PADDLE_PACKAGE", True)

DEFAULT_RUN_EXPERIMENT_NAMES = [
    "picodet_l_320_baseline_fast_20e",
    "picodet_l_640_scale_fast_20e",
    "picodet_l_640_oversample_fast_20e",
    "picodet_l_640_zoom_crop_fast_20e",
]
RUN_EXPERIMENT_NAMES = [
    name.replace("_fast_20e", f"_{TRAIN_AUG_PROFILE}_20e")
    for name in DEFAULT_RUN_EXPERIMENT_NAMES
]

EXPECTED_CLASS_NAMES = [
    "person",
    "ear",
    "ear-mufs",
    "face",
    "face-guard",
    "face-mask",
    "foot",
    "tool",
    "glasses",
    "gloves",
    "helmet",
    "hands",
    "head",
    "medical-suit",
    "shoes",
    "safety-suit",
    "safety-vest",
]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("PYTHON_EXECUTABLE:", sys.executable)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("PADDLEDET_ROOT:", PADDLEDET_ROOT)
print("RUN_TRAINING:", RUN_TRAINING, "RUN_SPEED_BENCHMARK:", RUN_SPEED_BENCHMARK)
print("USE_GPU:", USE_GPU, "ALLOW_CPU_FALLBACK:", ALLOW_CPU_FALLBACK, "TRAIN_WITH_AMP:", TRAIN_WITH_AMP)
print("VERIFY_PADDLE_GPU_RUNTIME:", VERIFY_PADDLE_GPU_RUNTIME)
print("PADDLE_INSTALL_PACKAGE:", PADDLE_INSTALL_PACKAGE, "PADDLE_INSTALL_INDEX_URL:", PADDLE_INSTALL_INDEX_URL or "<default PyPI>")
print("TRAIN_AUG_PROFILE:", TRAIN_AUG_PROFILE)
print("PILOT_EPOCHS:", PILOT_EPOCHS, "FULL_EPOCHS:", FULL_EPOCHS, "EPOCHS:", EPOCHS)
print("SNAPSHOT_EPOCH:", SNAPSHOT_EPOCH, "WORKER_NUM:", WORKER_NUM, "BENCHMARK_IMAGE_LIMIT:", BENCHMARK_IMAGE_LIMIT)


PYTHON_EXECUTABLE: D:\Anaconda\envs\SH17-relieq\python.exe
DATA_ROOT: E:\data\SH17
OUTPUT_ROOT: D:\DS-AI\SH17\outputs\picodet_l_paddledet
PADDLEDET_ROOT: D:\DS-AI\SH17\outputs\picodet_l_paddledet\PaddleDetection
RUN_TRAINING: False RUN_SPEED_BENCHMARK: True
USE_GPU: True ALLOW_CPU_FALLBACK: True TRAIN_WITH_AMP: True
VERIFY_PADDLE_GPU_RUNTIME: True
PADDLE_INSTALL_PACKAGE: paddlepaddle PADDLE_INSTALL_INDEX_URL: <default PyPI>
TRAIN_AUG_PROFILE: fast
PILOT_EPOCHS: 20 FULL_EPOCHS: 50 EPOCHS: 20
SNAPSHOT_EPOCH: 5 WORKER_NUM: 6 BENCHMARK_IMAGE_LIMIT: 960


In [2]:
import sys
from pathlib import Path


def _helper_roots():
    return [
        PROJECT_ROOT,
        PROJECT_ROOT / "SH17",
        Path.cwd(),
        Path.cwd() / "SH17",
    ]


def _add_helper_root_to_path():
    checked_paths = []
    for root in _helper_roots():
        helper_path = root / "scripts" / "sh17_picodet_dataset.py"
        checked_paths.append(helper_path)
        if helper_path.exists():
            root_text = str(root)
            if root_text not in sys.path:
                sys.path.insert(0, root_text)
            return helper_path

    checked_text = "\n".join(f"  - {path}" for path in checked_paths)
    raise FileNotFoundError(
        "Cannot find scripts/sh17_picodet_dataset.py.\n"
        "Run this notebook from the SH17 project folder, or upload/clone the full SH17 folder next to the notebook.\n"
        f"Checked:\n{checked_text}"
    )


HELPER_MODULE_PATH = _add_helper_root_to_path()
print("Helper module:", HELPER_MODULE_PATH)

from scripts.sh17_picodet_dataset import (
    CLASS_NAMES,
    MINORITY_CLASS_IDS_ZERO_BASED,
    MINORITY_REPEAT_FACTOR,
    PICODET_L_PARAMS_M,
    build_benchmark_coco_subset,
    build_oversampled_coco,
    convert_manifest_to_coco,
    parse_picodet_speed_log,
    picodet_experiments,
    picodet_speed_benchmark_experiments,
    write_picodet_configs,
)

assert CLASS_NAMES == EXPECTED_CLASS_NAMES, "Class order drifted from the SH17 presentation setup."
print(f"PP-PicoDet-L params: {PICODET_L_PARAMS_M:.2f}M")
print("Minority classes:", [CLASS_NAMES[index] for index in sorted(MINORITY_CLASS_IDS_ZERO_BASED)])


Helper module: D:\DS-AI\SH17\scripts\sh17_picodet_dataset.py
PP-PicoDet-L params: 5.80M
Minority classes: ['ear-mufs', 'face-guard', 'foot', 'helmet', 'medical-suit', 'safety-vest']


In [3]:
import importlib.util
import importlib.metadata as importlib_metadata
import random
import subprocess
import textwrap
from collections import deque
from datetime import datetime


def _command_log_path(log_name):
    if log_name is None:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        log_name = f"command_{stamp}.log"
    return LOG_DIR / log_name


def command_env(extra_env=None):
    base_env = os.environ.copy()
    base_env.setdefault("PYTHONUTF8", "1")
    base_env.setdefault("PYTHONIOENCODING", "utf-8")
    if extra_env:
        base_env.update(extra_env)
    return base_env


def run_command(command, cwd=None, check=True, env=None, log_name=None):
    log_path = _command_log_path(log_name)
    print("\n$", " ".join(str(part) for part in command))
    print("log:", log_path)
    merged_env = command_env(env)
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        text=True,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        encoding="utf-8",
        errors="replace",
    )
    with log_path.open("w", encoding="utf-8", errors="replace") as handle:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            handle.write(line)
            tail.append(line.rstrip())
    returncode = process.wait()
    result = subprocess.CompletedProcess(command, returncode)
    result.log_path = log_path
    result.tail = list(tail)
    if check and returncode != 0:
        env_note = ""
        if env is not None:
            interesting = ["CUDA_VISIBLE_DEVICES", "PADDLE_INSTALL_PACKAGE", "SH17_DATA_ROOT"]
            env_note = "\nEnvironment overrides: " + ", ".join(
                f"{name}={env.get(name)}" for name in interesting if env.get(name) is not None
            )
        tail_text = "\n".join(result.tail[-40:])
        raise RuntimeError(
            textwrap.dedent(
                f"""
                Command failed with exit code {returncode}
                Command: {command}
                CWD: {cwd}
                Log: {log_path}{env_note}

                Last output lines:
                {tail_text}
                """
            ).strip()
        )
    return result


def pip_install_env():
    pip_env = command_env()
    # PaddleDetection requirements still include sklearn==0.0. Newer pip blocks it
    # unless this compatibility flag is set.
    pip_env.setdefault("SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL", "True")
    return pip_env


def ensure_supported_requirements_python():
    if sys.version_info >= (3, 13):
        raise RuntimeError(
            textwrap.dedent(
                f"""
                This notebook is running Python {sys.version.split()[0]}, but PaddleDetection requirements pin numpy<2.0.
                NumPy 1.26.x does not provide compatible Python 3.13 wheels, so pip falls back to building NumPy
                from source on Windows/Anaconda and fails with compiler errors such as GCC >= 8.4.

                Create a fresh training environment with Python 3.12, then rerun the notebook. Example:
                    conda create -n SH17 python=3.12 -y
                    conda activate SH17
                    python -m pip install --upgrade pip
                """
            ).strip()
        )


def paddledet_requirements_install_command(requirements):
    command = [sys.executable, "-m", "pip", "install"]
    if PADDLEDET_REQUIREMENTS_ONLY_BINARY:
        command.append("--only-binary=" + ",".join(PADDLEDET_BINARY_ONLY_PACKAGES))
    command.extend(["-r", str(requirements)])
    return command


def ensure_bootstrap_packages():
    missing_pkg_resources = importlib.util.find_spec("pkg_resources") is None
    missing_wheel = importlib.util.find_spec("wheel") is None
    if missing_pkg_resources or missing_wheel:
        pip_env = pip_install_env()
        run_command([sys.executable, "-m", "pip", "install", *BOOTSTRAP_PIP_PACKAGES], env=pip_env)


def installed_paddle_packages():
    names = set()
    for distribution in importlib_metadata.distributions():
        name = distribution.metadata.get("Name", "").lower()
        if name in {"paddlepaddle", "paddlepaddle-gpu"}:
            names.add(name)
    return names


def _install_paddle_package():
    command = [sys.executable, "-m", "pip", "install", PADDLE_INSTALL_PACKAGE]
    if PADDLE_INSTALL_INDEX_URL:
        command.extend(["-i", PADDLE_INSTALL_INDEX_URL])
    run_command(command, env=pip_install_env())


def ensure_paddle_package():
    installed = installed_paddle_packages()
    desired = PADDLE_INSTALL_PACKAGE.lower()
    wants_gpu = "paddlepaddle-gpu" in desired
    wants_cpu = desired == "paddlepaddle"
    has_cpu_wheel = "paddlepaddle" in installed
    has_gpu_wheel = "paddlepaddle-gpu" in installed

    if wants_cpu and has_gpu_wheel and AUTO_REINSTALL_PADDLE_PACKAGE:
        if "paddle" in sys.modules:
            raise RuntimeError("Paddle is already imported. Restart the kernel before switching from paddlepaddle-gpu to paddlepaddle.")
        run_command([sys.executable, "-m", "pip", "uninstall", "-y", "paddlepaddle-gpu"], env=pip_install_env())
        _install_paddle_package()
    elif wants_gpu and has_cpu_wheel and not has_gpu_wheel and AUTO_REINSTALL_PADDLE_PACKAGE:
        if "paddle" in sys.modules:
            raise RuntimeError("Paddle is already imported. Restart the kernel before switching from paddlepaddle to paddlepaddle-gpu.")
        run_command([sys.executable, "-m", "pip", "uninstall", "-y", "paddlepaddle"], env=pip_install_env())
        _install_paddle_package()
    elif wants_gpu and not has_gpu_wheel:
        _install_paddle_package()
    elif wants_cpu and not has_cpu_wheel:
        _install_paddle_package()
    elif not installed:
        _install_paddle_package()


def verify_paddle_gpu_runtime(paddle, device):
    if not VERIFY_PADDLE_GPU_RUNTIME:
        print("VERIFY_PADDLE_GPU_RUNTIME=False: skipping Paddle GPU runtime check.")
        return
    if not USE_GPU or "gpu" not in str(device).lower():
        return

    print("Verifying Paddle GPU runtime with paddle.utils.run_check() and a small Conv2D op...")
    try:
        paddle.utils.run_check()
        x = paddle.randn([1, 3, 16, 16], dtype="float32")
        conv = paddle.nn.Conv2D(3, 4, 3)
        y = conv(x)
        _ = y.numpy()
    except Exception as exc:
        raise RuntimeError(
            textwrap.dedent(
                f"""
                Paddle detected GPU device {device}, but the GPU runtime check failed before training.

                Current Python executable:
                    {sys.executable}

                Most likely causes on Windows:
                1. The notebook kernel is not the intended conda env. Your train command should point to
                   something like D:\\Anaconda\\envs\\SH17\\python.exe, not D:\\Anaconda\\python.exe.
                2. The installed Paddle GPU wheel cannot find a matching CUDA/cuDNN runtime, commonly shown as
                   cudnn64_8.dll missing or not configured correctly.

                Recommended fix for this project on the 4060 Ti machine:
                    conda activate SH17
                    python -m pip uninstall -y paddlepaddle paddlepaddle-gpu
                    python -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
                    python -m pip install ipykernel
                    python -m ipykernel install --user --name SH17 --display-name "Python (SH17)"

                Then restart Jupyter, select kernel "Python (SH17)", rerun setup, and confirm PYTHON_EXECUTABLE.

                Original error:
                {exc}
                """
            ).strip()
        ) from exc


def ensure_paddledetection():
    if AUTO_SETUP and not PADDLEDET_ROOT.exists():
        PADDLEDET_ROOT.parent.mkdir(parents=True, exist_ok=True)
        run_command([
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/PaddlePaddle/PaddleDetection.git",
            str(PADDLEDET_ROOT),
        ], log_name="clone_paddledetection.log")

    if AUTO_SETUP:
        ensure_bootstrap_packages()
        ensure_paddle_package()

    if AUTO_SETUP and INSTALL_PADDLEDET_REQUIREMENTS:
        ensure_supported_requirements_python()
        requirements = PADDLEDET_ROOT / "requirements.txt"
        if requirements.exists():
            pip_env = pip_install_env()
            try:
                run_command(paddledet_requirements_install_command(requirements), env=pip_env)
            except RuntimeError as exc:
                raise RuntimeError(
                    textwrap.dedent(
                        """
                        PaddleDetection requirements installation failed.
                        The full pip log is printed above. Common local Windows/Anaconda causes are
                        incompatible Python versions, pycocotools/opencv wheels, or the deprecated sklearn package.
                        On Windows/Anaconda, use Python 3.12 so numpy<2.0 has a prebuilt wheel.

                        You can install the missing package manually, then set:
                            INSTALL_PADDLEDET_REQUIREMENTS = False
                        and rerun this cell.
                        """
                    ).strip()
                ) from exc
    elif AUTO_SETUP:
        print("INSTALL_PADDLEDET_REQUIREMENTS=False: skipping PaddleDetection requirements install.")

    try:
        import paddle
    except Exception:
        if not AUTO_SETUP:
            raise
        print(f"Paddle import failed. Trying configured install package: {PADDLE_INSTALL_PACKAGE}")
        _install_paddle_package()
        import paddle

    random.seed(SEED)
    paddle.seed(SEED)
    device = paddle.device.get_device()
    print("Paddle device:", device)
    if USE_GPU and "gpu" not in device.lower():
        if ALLOW_CPU_FALLBACK:
            print("USE_GPU=True but Paddle does not report a GPU device. Falling back to CPU.")
            paddle.set_device("cpu")
            device = paddle.device.get_device()
        else:
            raise RuntimeError(
                textwrap.dedent(
                    """
                    USE_GPU=True but Paddle does not report a GPU device.
                    Install a PaddlePaddle GPU wheel matching the machine CUDA version, then rerun this cell,
                    or set SH17_ALLOW_CPU_FALLBACK=1 to continue CPU smoke checks.
                    """
                ).strip()
            )
    elif not USE_GPU:
        paddle.set_device("cpu")
        device = paddle.device.get_device()
    verify_paddle_gpu_runtime(paddle, device)
    return device


if AUTO_SETUP:
    PADDLE_DEVICE = ensure_paddledetection()
else:
    PADDLE_DEVICE = "gpu" if USE_GPU else "cpu"
    print("AUTO_SETUP=False: skipping dependency setup.")

ACTIVE_USE_GPU = "gpu" in str(PADDLE_DEVICE).lower()
print("ACTIVE_USE_GPU:", ACTIVE_USE_GPU)



$ D:\Anaconda\envs\SH17-relieq\python.exe -m pip uninstall -y paddlepaddle-gpu
log: D:\DS-AI\SH17\outputs\picodet_l_paddledet\logs\command_20260618_003402_994468.log
Found existing installation: paddlepaddle-gpu 3.3.1
Uninstalling paddlepaddle-gpu-3.3.1:
  Successfully uninstalled paddlepaddle-gpu-3.3.1

$ D:\Anaconda\envs\SH17-relieq\python.exe -m pip install paddlepaddle
log: D:\DS-AI\SH17\outputs\picodet_l_paddledet\logs\command_20260618_003404_175760.log
  Using cached paddlepaddle-3.3.1-cp312-cp312-win_amd64.whl.metadata (8.8 kB)
Using cached paddlepaddle-3.3.1-cp312-cp312-win_amd64.whl (104.8 MB)

$ D:\Anaconda\envs\SH17-relieq\python.exe -m pip install --only-binary=numpy,scipy,opencv-python,pycocotools,shapely -r D:\DS-AI\SH17\outputs\picodet_l_paddledet\PaddleDetection\requirements.txt
log: D:\DS-AI\SH17\outputs\picodet_l_paddledet\logs\command_20260618_003419_188505.log


D:\Anaconda\envs\SH17-relieq\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Paddle device: cpu
USE_GPU=True but Paddle does not report a GPU device. Falling back to CPU.
ACTIVE_USE_GPU: False


In [4]:
def _validate_manifest_paths():
    missing = []
    if not TRAIN_MANIFEST.exists():
        missing.append(f"train manifest: {TRAIN_MANIFEST}")
    if not VAL_MANIFEST.exists():
        missing.append(f"val manifest: {VAL_MANIFEST}")
    if not missing:
        return

    checked_roots = "\n".join(
        f"  - {root} -> train={root / 'train_files.txt'}, val={root / 'val_files.txt'}"
        for root in DATA_ROOT_CANDIDATES
    )
    raise FileNotFoundError(
        "Missing dataset manifests.\n"
        + "\n".join(f"  - {item}" for item in missing)
        + "\n\nSet DATA_ROOT to the SH17 dataset folder that contains train_files.txt and val_files.txt, "
        + "or set SH17_DATA_ROOT, SH17_TRAIN_MANIFEST, and SH17_VAL_MANIFEST explicitly.\n"
        + "Example local override:\n"
        + "  DATA_ROOT = Path(r'E:/data/SH17')\n"
        + "\nChecked common roots:\n"
        + checked_roots
    )


_validate_manifest_paths()

DATASET_ARTIFACT_DIR = OUTPUT_ROOT / "dataset"
ANNOTATION_DIR = DATASET_ARTIFACT_DIR / "annotations"
TRAIN_JSON = ANNOTATION_DIR / "instances_train.json"
VAL_JSON = ANNOTATION_DIR / "instances_val.json"
TRAIN_OVERSAMPLED_JSON = ANNOTATION_DIR / "instances_train_oversampled.json"
TRAIN_BENCHMARK_JSON = ANNOTATION_DIR / f"instances_train_benchmark_{BENCHMARK_IMAGE_LIMIT}.json"

train_stats = convert_manifest_to_coco(
    manifest_path=TRAIN_MANIFEST,
    dataset_root=DATA_ROOT,
    output_json=TRAIN_JSON,
)
val_stats = convert_manifest_to_coco(
    manifest_path=VAL_MANIFEST,
    dataset_root=DATA_ROOT,
    output_json=VAL_JSON,
)
oversampled_stats = build_oversampled_coco(
    train_json=TRAIN_JSON,
    output_json=TRAIN_OVERSAMPLED_JSON,
    minority_class_ids_zero_based=MINORITY_CLASS_IDS_ZERO_BASED,
    repeat_factor=MINORITY_REPEAT_FACTOR,
)
benchmark_stats = build_benchmark_coco_subset(
    source_json=TRAIN_JSON,
    output_json=TRAIN_BENCHMARK_JSON,
    image_limit=BENCHMARK_IMAGE_LIMIT,
)

print("Train COCO:", train_stats)
print("Val COCO:", val_stats)
print("Oversampled train COCO:", oversampled_stats)
print("Benchmark train COCO:", benchmark_stats)
print("Annotations:", ANNOTATION_DIR)


Train COCO: {'images': 6479, 'instances': 60636, 'categories': 17}
Val COCO: {'images': 1620, 'instances': 15358, 'categories': 17}
Oversampled train COCO: {'images': 8483, 'instances': 88944, 'categories': 17}
Benchmark train COCO: {'images': 960, 'instances': 8792, 'categories': 17}
Annotations: D:\DS-AI\SH17\outputs\picodet_l_paddledet\dataset\annotations


In [5]:
CONFIG_DIR = OUTPUT_ROOT / "configs"
RUNS_DIR = OUTPUT_ROOT / "runs"
TRAIN_EXPERIMENTS = picodet_experiments(TRAIN_AUG_PROFILE)
CONFIG_PATHS = write_picodet_configs(
    config_dir=CONFIG_DIR,
    dataset_dir=DATA_ROOT,
    output_dir=RUNS_DIR,
    paddledet_root=PADDLEDET_ROOT,
    epochs=EPOCHS,
    snapshot_epoch=SNAPSHOT_EPOCH,
    worker_num=WORKER_NUM,
    use_shared_memory=USE_SHARED_MEMORY,
    use_gpu=ACTIVE_USE_GPU,
    experiments=TRAIN_EXPERIMENTS,
)

EXPERIMENTS = {experiment.name: experiment for experiment in TRAIN_EXPERIMENTS}
CONFIG_BY_EXPERIMENT = {experiment.name: CONFIG_DIR / experiment.config_file_name for experiment in EXPERIMENTS.values()}

missing_experiments = [name for name in RUN_EXPERIMENT_NAMES if name not in EXPERIMENTS]
if missing_experiments:
    raise KeyError(f"Unknown experiment names: {missing_experiments}")

for experiment_name in RUN_EXPERIMENT_NAMES:
    experiment = EXPERIMENTS[experiment_name]
    config_path = CONFIG_BY_EXPERIMENT[experiment_name]
    print(
        f"{experiment_name}: input={experiment.input_size}, batch={experiment.batch_size}, "
        f"lr={experiment.base_lr}, aug={experiment.augmentation_profile}, train_json={experiment.train_json_name}, config={config_path}"
    )


picodet_l_320_baseline_fast_20e: input=320, batch=24, lr=0.12, aug=fast, train_json=instances_train.json, config=D:\DS-AI\SH17\outputs\picodet_l_paddledet\configs\picodet_l_320_fast_sh17.yml
picodet_l_640_scale_fast_20e: input=640, batch=12, lr=0.06, aug=fast, train_json=instances_train.json, config=D:\DS-AI\SH17\outputs\picodet_l_paddledet\configs\picodet_l_640_fast_sh17.yml
picodet_l_640_oversample_fast_20e: input=640, batch=12, lr=0.06, aug=fast, train_json=instances_train_oversampled.json, config=D:\DS-AI\SH17\outputs\picodet_l_paddledet\configs\picodet_l_640_oversample_fast_sh17.yml
picodet_l_640_zoom_crop_fast_20e: input=640, batch=12, lr=0.06, aug=fast, train_json=instances_train.json, config=D:\DS-AI\SH17\outputs\picodet_l_paddledet\configs\picodet_l_640_zoom_crop_fast_sh17.yml


In [6]:
import csv

SPEED_BENCHMARK_ROOT = OUTPUT_ROOT / "speed_benchmarks"
SPEED_BENCHMARK_CONFIG_DIR = SPEED_BENCHMARK_ROOT / "configs"
SPEED_BENCHMARK_RUNS_DIR = SPEED_BENCHMARK_ROOT / "runs"
SPEED_BENCHMARK_CSV = SPEED_BENCHMARK_ROOT / "speed_benchmark.csv"
BENCHMARK_EXPERIMENTS = picodet_speed_benchmark_experiments(
    input_size=320,
    train_json_name=TRAIN_BENCHMARK_JSON.name,
)
BENCHMARK_CONFIG_PATHS = write_picodet_configs(
    config_dir=SPEED_BENCHMARK_CONFIG_DIR,
    dataset_dir=DATA_ROOT,
    output_dir=SPEED_BENCHMARK_RUNS_DIR,
    paddledet_root=PADDLEDET_ROOT,
    epochs=1,
    snapshot_epoch=1,
    worker_num=WORKER_NUM,
    use_shared_memory=USE_SHARED_MEMORY,
    use_gpu=ACTIVE_USE_GPU,
    experiments=BENCHMARK_EXPERIMENTS,
)
BENCHMARK_CONFIG_BY_EXPERIMENT = {
    experiment.name: SPEED_BENCHMARK_CONFIG_DIR / experiment.config_file_name
    for experiment in BENCHMARK_EXPERIMENTS
}


def train_speed_benchmark(experiment):
    config_path = BENCHMARK_CONFIG_BY_EXPERIMENT[experiment.name]
    command = [
        sys.executable,
        "tools/train.py",
        "-c",
        str(config_path),
    ]
    if TRAIN_WITH_AMP and ACTIVE_USE_GPU:
        command.append("--amp")
    return run_command(command, cwd=PADDLEDET_ROOT, log_name=f"speed_benchmark_{experiment.augmentation_profile}.log")


def write_speed_benchmark_csv():
    SPEED_BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
    rows = []
    for experiment in BENCHMARK_EXPERIMENTS:
        log_path = LOG_DIR / f"speed_benchmark_{experiment.augmentation_profile}.log"
        metrics = parse_picodet_speed_log(log_path)
        row = {
            "profile": experiment.augmentation_profile,
            "experiment": experiment.name,
            "input_size": experiment.input_size,
            "batch_size": experiment.batch_size,
            "worker_num": WORKER_NUM,
            "use_shared_memory": USE_SHARED_MEMORY,
            "config": str(BENCHMARK_CONFIG_BY_EXPERIMENT[experiment.name]),
            **metrics,
        }
        rows.append(row)
    fieldnames = list(rows[0].keys())
    with SPEED_BENCHMARK_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return rows


if RUN_SPEED_BENCHMARK:
    for benchmark_experiment in BENCHMARK_EXPERIMENTS:
        train_speed_benchmark(benchmark_experiment)
    speed_rows = write_speed_benchmark_csv()
    print("Wrote speed benchmark:", SPEED_BENCHMARK_CSV)
    for row in speed_rows:
        print(row)
else:
    print("RUN_SPEED_BENCHMARK=False. Set it True to compare augmentation-profile data_cost before full training.")
    print("Benchmark configs:", SPEED_BENCHMARK_CONFIG_DIR)



$ D:\Anaconda\envs\SH17-relieq\python.exe tools/train.py -c D:\DS-AI\SH17\outputs\picodet_l_paddledet\speed_benchmarks\configs\bench_picodet_l_320_official.yml
log: D:\DS-AI\SH17\outputs\picodet_l_paddledet\logs\speed_benchmark_official.log
INFO: Could not find files for the given pattern(s).
[06/18 00:39:40] ppdet.utils.checkpoint INFO: Skipping import of the encryption module.
======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_enable_pir_in_executor', current_value=True, default_value=False)
[06/18 00:39:40] ppdet.utils.download WARNING: Config annotation D:/DS-AI/SH17/outputs/picodet_l_paddledet/speed_benchmarks/dataset/annotations/instances_train_benchmark_960.json is not a file, dataset config is not valid
Traceback (most recent call last):
  File "D:\DS-AI\SH17\outputs\picodet_l_paddledet\PaddleDetection\tools\train.py", line 219, in <module>
    main()
  File "D:\DS-AI\SH17\outputs\picodet_l_paddledet\PaddleDetection\tools\train.py", line 

RuntimeError: Command failed with exit code 1
                Command: ['D:\\Anaconda\\envs\\SH17-relieq\\python.exe', 'tools/train.py', '-c', 'D:\\DS-AI\\SH17\\outputs\\picodet_l_paddledet\\speed_benchmarks\\configs\\bench_picodet_l_320_official.yml']
                CWD: D:\DS-AI\SH17\outputs\picodet_l_paddledet\PaddleDetection
                Log: D:\DS-AI\SH17\outputs\picodet_l_paddledet\logs\speed_benchmark_official.log

                Last output lines:
                INFO: Could not find files for the given pattern(s).
Warning: Unable to use numba in PP-Tracking, please install numba, for example(python3.7): `pip install numba==0.56.4`
Warning: Unable to use numba in PP-Tracking, please install numba, for example(python3.7): `pip install numba==0.56.4`
[06/18 00:39:40] ppdet.utils.checkpoint INFO: Skipping import of the encryption module.
Warning: import ppdet from source directory without installing, run 'python setup.py install' to install ppdet firstly
=====================

In [ ]:
def _checkpoint_exists(base_path):
    return base_path.exists() or base_path.with_suffix(".pdparams").exists()


def _checkpoint_search_dirs(experiment_name):
    run_dir = RUNS_DIR / experiment_name
    config_stem = CONFIG_BY_EXPERIMENT[experiment_name].stem
    candidates = [run_dir, run_dir / config_stem]
    if run_dir.exists():
        candidates.extend(path for path in run_dir.iterdir() if path.is_dir())
    seen = set()
    ordered = []
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            ordered.append(path)
            seen.add(key)
    return ordered


def _numeric_checkpoint_bases(run_dir):
    checkpoints = []
    if not run_dir.exists():
        return checkpoints
    for checkpoint_path in run_dir.glob("*.pdparams"):
        if checkpoint_path.stem.isdigit():
            checkpoints.append((int(checkpoint_path.stem), checkpoint_path.with_suffix("")))
    return [base_path for _, base_path in sorted(checkpoints, reverse=True)]


def checkpoint_base_path(experiment_name):
    candidates = []
    for run_dir in _checkpoint_search_dirs(experiment_name):
        candidates.extend([
            run_dir / "best_model",
            run_dir / "model_final",
            run_dir / "last_model",
            *_numeric_checkpoint_bases(run_dir),
        ])
    for candidate in candidates:
        if _checkpoint_exists(candidate):
            return candidate
    return RUNS_DIR / experiment_name / "best_model"


def last_checkpoint_base_path(experiment_name):
    candidates = []
    for run_dir in _checkpoint_search_dirs(experiment_name):
        candidates.extend([
            run_dir / "last_model",
            run_dir / "model_final",
            *_numeric_checkpoint_bases(run_dir),
            run_dir / "best_model",
        ])
    for candidate in candidates:
        if _checkpoint_exists(candidate):
            return candidate
    return None


def train_experiment(experiment_name):
    config_path = CONFIG_BY_EXPERIMENT[experiment_name]
    run_dir = RUNS_DIR / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable,
        "tools/train.py",
        "-c",
        str(config_path),
        "--eval",
    ]
    if TRAIN_WITH_AMP and ACTIVE_USE_GPU:
        command.append("--amp")
    last_checkpoint = last_checkpoint_base_path(experiment_name)
    if last_checkpoint is not None:
        command.extend(["-r", str(last_checkpoint)])
    return run_command(command, cwd=PADDLEDET_ROOT, log_name=f"train_{experiment_name}.log")


def evaluate_experiment(experiment_name):
    config_path = CONFIG_BY_EXPERIMENT[experiment_name]
    eval_dir = OUTPUT_ROOT / "eval" / experiment_name
    eval_dir.mkdir(parents=True, exist_ok=True)
    weights = checkpoint_base_path(experiment_name)
    command = [
        sys.executable,
        "tools/eval.py",
        "-c",
        str(config_path),
        "-o",
        f"weights={weights}",
        "--classwise",
        "--output_eval",
        str(eval_dir),
    ]
    return run_command(command, cwd=PADDLEDET_ROOT, log_name=f"eval_{experiment_name}.log")


if RUN_TRAINING:
    for experiment_name in RUN_EXPERIMENT_NAMES:
        train_experiment(experiment_name)
        evaluate_experiment(experiment_name)
else:
    print("RUN_TRAINING=False. Review configs and set RUN_TRAINING=True on the training machine.")


In [ ]:
import csv
import re

SUMMARY_METRICS_CSV = OUTPUT_ROOT / "summary_metrics.csv"
PER_CLASS_METRICS_CSV = OUTPUT_ROOT / "per_class_metrics.csv"


COCO_METRIC_PATTERNS = {
    "map50_95": r"Average Precision\s+\(AP\).*IoU=0\.50:0\.95.*=\s*([0-9.]+)",
    "map50": r"Average Precision\s+\(AP\).*IoU=0\.50\s+\|.*=\s*([0-9.]+)",
    "mar100": r"Average Recall\s+\(AR\).*maxDets=100.*=\s*([0-9.]+)",
}


def _latest_eval_log(experiment_name):
    matches = sorted(LOG_DIR.glob(f"eval_{experiment_name}*.log"), key=lambda path: path.stat().st_mtime)
    return matches[-1] if matches else None


def _parse_coco_metrics(log_path):
    metrics = {"map50_95": "pending", "map50": "pending", "precision": "pending", "recall": "pending"}
    if log_path is None or not log_path.exists():
        return metrics, "pending"
    text = log_path.read_text(encoding="utf-8", errors="replace")
    found_any = False
    for key, pattern in COCO_METRIC_PATTERNS.items():
        match = re.search(pattern, text)
        if match:
            found_any = True
            value = f"{float(match.group(1)):.6f}"
            if key == "mar100":
                metrics["recall"] = value
            else:
                metrics[key] = value
    status = "real" if found_any else "pending"
    return metrics, status


def write_summary_metrics():
    rows = []
    for experiment_name in RUN_EXPERIMENT_NAMES:
        experiment = EXPERIMENTS[experiment_name]
        eval_dir = OUTPUT_ROOT / "eval" / experiment_name
        metrics, status = _parse_coco_metrics(_latest_eval_log(experiment_name))
        row = {
            "experiment": experiment_name,
            "input_size": experiment.input_size,
            "params_m": PICODET_L_PARAMS_M,
            "batch_size": experiment.batch_size,
            "base_lr": experiment.base_lr,
            "epochs": EPOCHS,
            "epochs_trained": EPOCHS,
            "augmentation_profile": experiment.augmentation_profile,
            "train_json": experiment.train_json_name,
            "map50_95": metrics["map50_95"],
            "map50": metrics["map50"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "status": status,
            "bbox_json": str(eval_dir / "bbox.json"),
        }
        rows.append(row)
    with SUMMARY_METRICS_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return rows


def write_pending_per_class_metrics():
    rows = []
    for experiment_name in RUN_EXPERIMENT_NAMES:
        for class_name in CLASS_NAMES:
            rows.append({
                "experiment": experiment_name,
                "class_name": class_name,
                "ap": "pending",
                "precision": "pending",
                "recall": "pending",
                "status": "pending",
            })
    with PER_CLASS_METRICS_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return rows


summary_rows = write_summary_metrics()
per_class_rows = write_pending_per_class_metrics()

print("Wrote summary metrics:", SUMMARY_METRICS_CSV)
print("Wrote per-class metrics scaffold:", PER_CLASS_METRICS_CSV)
for row in summary_rows:
    print(row)


## Training Notes

Recommended run order for the 4060 Ti 16GB Windows machine:

1. Run setup, conversion, and config cells with `RUN_TRAINING = False` and `RUN_SPEED_BENCHMARK = False`.
2. Set `RUN_SPEED_BENCHMARK = True`, keep `RUN_TRAINING = False`, rerun config + benchmark cells, and inspect `outputs/picodet_l_paddledet/speed_benchmarks/speed_benchmark.csv`.
3. Keep `TRAIN_AUG_PROFILE = "fast"` unless the benchmark clearly favors another profile.
4. Set `RUN_TRAINING = True` for the selected 20-epoch pilot matrix.
5. If a variant looks promising, set `SH17_EPOCHS=50` and resume that variant to full training.

Missing metrics should remain `pending` until a real eval log exists. Do not fill report metrics manually.
